# 🔭 Conesearch — Fink/LSST

Ce notebook effectue une **recherche par position** dans la base Fink/LSST :
donner un centre (RA, Dec) et un rayon en arcsecondes, et explorer tous les
objets Fink dans ce champ.

Pour chaque objet trouvé :
- Position sur une **carte locale** (projection gnomonique centrée sur la cible)
- **Courbes de lumière** (flux et magnitude AB)
- **Cutouts** Science / Template / Difference de la dernière alerte
- **Scores de classification** Fink disponibles
- **Tableau récapitulatif** avec liens vers le portail

Le centre peut être spécifié en degrés décimaux **ou** en sexagésimal,
ou par **résolution de nom** (ex. `"NGC 1365"`) via l'API Fink.

**API :** `https://api.lsst.fink-portal.org`  
Endpoints : `/api/v1/conesearch` · `/api/v1/sources` · `/api/v1/cutouts` · `/api/v1/resolver`

## 1. Paramètres

In [ ]:
# ─── Centre du champ ──────────────────────────────────────────────────────────
# Option A : coordonnées décimales
TARGET_RA   = 53.16    # degrés décimaux
TARGET_DEC  = -28.10  # degrés décimaux
TARGET_NAME = None        # ex. "NGC 1365" — si renseigné, résout le nom et ignore RA/Dec

# ─── Rayon de recherche ───────────────────────────────────────────────────────
RADIUS_ARCSEC = 1800.0     # arcsec (max 18000 = 5 deg)

# ─── Bornes temporelles (optionnel) ───────────────────────────────────────────
STARTDATE  = None         # "2025-10-01 00:00:00" ou None
STOPDATE   = None         # "2026-03-15 00:00:00" ou None
#STARTDATE  = "2025-10-01 00:00:00"                     # "2026-01-01 00:00:00" ou None
#STOPDATE   = "2026-03-01 00:00:00"                    # "2026-03-01 00:00:00" ou None

# Alternative : WINDOW = 30  # jours depuis STARTDATE

# Minimum number of alerts (diaSources) per sky objects (diaObjectsId)
MIN_NUMBER_SOURCES = 10  # intger or None (no cut)

# Maximum sky objects to take into accounts
MAX_OBJECTS = 10

# ─── Options d'affichage ──────────────────────────────────────────────────────
SHOW_LIGHTCURVES = False    # Courbes de lumière pour chaque objet
SHOW_CUTOUTS     = False   # Cutouts pour chaque objet
SHOW_SCORES      = True    # Scores de classification
CUTOUT_CMAP      = 'hot'
CUTOUT_FORMAT    = 'PNG'
CUTOUT_KINDS     = ['Science', 'Template', 'Difference']

BASE_URL   = "https://api.lsst.fink-portal.org"
SAVE_FIGS  = False
OUTPUT_DIR = "conesearch_outputs"
# ─────────────────────────────────────────────────────────────────────────────

## 2. Imports

In [ ]:
import os
import warnings
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
from io import BytesIO
from PIL import Image
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from IPython.display import display as ipy_display, HTML

warnings.filterwarnings('ignore')
os.makedirs(OUTPUT_DIR, exist_ok=True)

mpl.rcParams.update({
    'font.size'        : 11,
    'axes.titlesize'   : 12,
    'axes.titleweight' : 'bold',
    'axes.labelsize'   : 11,
    'xtick.labelsize'  : 9,
    'ytick.labelsize'  : 9,
    'legend.fontsize'  : 9,
    'figure.titlesize' : 13,
    'figure.titleweight': 'bold',
})

FILTER_COLORS = {
    'u': '#7B2FBE', 'g': '#0077BB', 'r': '#33AA77',
    'i': '#DDAA33', 'z': '#BB5500', 'y': '#AA0000',
}
OBJ_PALETTE = plt.cm.tab10(np.linspace(0, 1, 10))

MJD_EPOCH = pd.Timestamp('1858-11-17', tz='UTC')
ZP_AB     = 2.5 * np.log10(3631e9)

def mjd_to_datetime(mjd_series):
    return MJD_EPOCH + pd.to_timedelta(
        pd.to_numeric(mjd_series, errors='coerce'), unit='D')

def mjd_to_datestr(mjd):
    try:
        return (MJD_EPOCH + pd.to_timedelta(float(mjd), unit='D')).strftime('%Y-%m-%d')
    except Exception:
        return str(mjd)

print("Imports OK ✓")

## 3. Résolution du nom / coordonnées cible

In [ ]:
def resolve_name(name):
    """Résout un nom d'objet en (RA, Dec) via l'API Fink resolver."""
    resp = requests.get(
        f"{BASE_URL}/api/v1/resolver",
        params={"resolver": "Simbad", "name": name},
        timeout=30
    )
    resp.raise_for_status()
    data = resp.json()
    if not data or 'ra' not in data[0]:
        raise ValueError(f"Nom '{name}' non résolu.")
    return float(data[0]['ra']), float(data[0]['dec'])


if TARGET_NAME:
    print(f"Résolution de '{TARGET_NAME}'...")
    TARGET_RA, TARGET_DEC = resolve_name(TARGET_NAME)
    print(f"  → RA = {TARGET_RA:.6f}°   Dec = {TARGET_DEC:+.6f}°")
else:
    print(f"Cible : RA = {TARGET_RA:.6f}°   Dec = {TARGET_DEC:+.6f}°")

print(f"Rayon : {RADIUS_ARCSEC:.1f} arcsec ({RADIUS_ARCSEC/3600:.3f} deg)")

## 4. Conesearch — objets dans le champ

In [ ]:
def angular_separation_deg(ra1, dec1, ra2, dec2):
    """Séparation angulaire en degrés (formule haversine)."""
    ra1, dec1, ra2, dec2 = map(np.deg2rad, [ra1, dec1, ra2, dec2])
    dra  = ra2 - ra1
    ddec = dec2 - dec1
    a = np.sin(ddec/2)**2 + np.cos(dec1)*np.cos(dec2)*np.sin(dra/2)**2
    return np.rad2deg(2 * np.arcsin(np.sqrt(a)))


payload = {
    "ra"    : str(TARGET_RA),
    "dec"   : str(TARGET_DEC),
    "radius": str(RADIUS_ARCSEC),
    "columns": "r:diaObjectId,r:ra,r:dec,r:band,r:psfFlux,r:midpointMjdTai,r:nDiaSources",
    "output-format": "json",
}
if STARTDATE: payload["startdate"] = STARTDATE
if STOPDATE:  payload["stopdate"]  = STOPDATE

resp = requests.post(f"{BASE_URL}/api/v1/conesearch", json=payload, timeout=60)
resp.raise_for_status()
data = resp.json()

if not data:
    print("⚠️  Aucun objet trouvé dans ce champ.")
    df_cone  = pd.DataFrame()
    object_ids = []
else:
    df_cone = pd.DataFrame(data)
    df_cone.columns = [c.replace('r:', '').replace('v:', '') for c in df_cone.columns]

    SKIP_COLS = {'band', 'diaObjectId', 'diaSourceId', 'nDiaSources'}
    for col in df_cone.columns:
        if col in SKIP_COLS:
            continue
        try:
            df_cone[col] = pd.to_numeric(df_cone[col], errors='coerce')
        except Exception:
            pass

    # Filter on number of sources
    if MIN_NUMBER_SOURCES is not None:
        df_cone.drop(df_cone[df_cone['nDiaSources'] < MIN_NUMBER_SOURCES].index, inplace = True)
        df_cone.reset_index(drop=True, inplace=True) # index de ligne remis à 0
        
    # Calcul local de la séparation
    df_cone['separation_degree'] = angular_separation_deg(
        TARGET_RA, TARGET_DEC,
        df_cone['ra'].values, df_cone['dec'].values
    )
    df_cone = df_cone.sort_values('separation_degree')

    id_col     = next(c for c in df_cone.columns if 'diaObjectId' in c)
    object_ids = df_cone[id_col].astype(str).unique().tolist()

    print(f"✓ {len(df_cone)} alertes trouvées pour {len(object_ids)} objet(s) with the criterias")
    print(f"  Séparation min : {df_cone['separation_degree'].min()*3600:.1f} arcsec")
    print(f"  Séparation max : {df_cone['separation_degree'].max()*3600:.1f} arcsec")
    ipy_display(df_cone[['diaObjectId','ra','dec','separation_degree','band','psfFlux','nDiaSources']].head(20))


## 5. Carte locale — projection gnomonique

Vue centrée sur la cible, axes en offset (ΔRA cos δ, ΔDec) en arcminutes.

In [ ]:
if df_cone.empty:
    print("⚠️  Aucun objet à afficher.")
else:    
    fig, ax = plt.subplots(figsize=(8, 8), layout='constrained')
    title_name = TARGET_NAME if TARGET_NAME else f"RA={TARGET_RA:.4f}° Dec={TARGET_DEC:+.4f}°"
    fig.suptitle(f"Champ Fink/LSST — {title_name}\n"
                 f"rayon {RADIUS_ARCSEC:.0f}" + "\" "
                 f"({len(object_ids)} objet(s))")

    # Offsets en arcminutes (projection gnomonic approx.)
    cos_dec = np.cos(np.deg2rad(TARGET_DEC))
    df_cone['dRA_arcmin']  = (df_cone['ra']  - TARGET_RA) * cos_dec * 60
    df_cone['dDec_arcmin'] = (df_cone['dec'] - TARGET_DEC) * 60

    # Un objet = une couleur
    for idx, oid in enumerate(object_ids[:1000]):
        sub   = df_cone[df_cone['diaObjectId'].astype(str) == oid]
        color = OBJ_PALETTE[idx % 10]
        ax.scatter(sub['dRA_arcmin'], sub['dDec_arcmin'],
                   color=color, s=10, zorder=5, alpha=0.85,
                   edgecolors='white', linewidths=0.4,
                   label=f"{str(oid)[:12]}...")
        # Label séparation
        sep_arcsec = sub['separation_degree'].iloc[0] * 3600
        #ax.annotate(f"{sep_arcsec:.1f}\"  ",
        #            xy=(sub['dRA_arcmin'].iloc[0], sub['dDec_arcmin'].iloc[0]),
        #            xytext=(4, 4), textcoords='offset points',
        #            fontsize=7, color=color)

    # Croix centrale = position cible
    ax.plot(0, 0, '+', color='black', ms=14, mew=2, zorder=10, label='Cible')

    # Cercle = rayon de recherche
    r_arcmin = RADIUS_ARCSEC / 60
    circle = mpatches.Circle((0, 0), r_arcmin, fill=False,
                               edgecolor='gray', lw=1.0, ls='--', zorder=3)
    ax.add_patch(circle)
    margin = r_arcmin * 1.15
    ax.set_xlim( margin, -margin)   # RA inversé (convention astronomique)
    ax.set_ylim(-margin,  margin)

    ax.set_xlabel("ΔRA cos δ (arcmin)")
    ax.set_ylabel("ΔDec (arcmin)")
    ax.set_aspect('equal')
    #ax.legend(fontsize=8, loc='upper right', framealpha=0.8)
    ax.grid(True, alpha=0.2, linewidth=0.5)

    if SAVE_FIGS:
        fig.savefig(f"{OUTPUT_DIR}/conesearch_map.pdf", bbox_inches='tight', dpi=150)
        fig.savefig(f"{OUTPUT_DIR}/conesearch_map.png", bbox_inches='tight', dpi=150)
        print(f"✓ Sauvegardé : {OUTPUT_DIR}/conesearch_map")
    plt.show()

## 6. Courbes de lumière + cutouts par objet

Pour chaque objet trouvé : figure de synthèse identique à `fink_lsst_alert_summary.ipynb`.

In [ ]:
# ── API + fonctions de tracé courbes de lumière (style fink_lsst_lightcurves_ddf) ──

FILTER_ORDER = ['u', 'g', 'r', 'i', 'z', 'y']  # ordre naturel LSST


def fetch_lightcurve(dia_object_id):
    """
    Télécharge la courbe de lumière complète d'un diaObjectId via /api/v1/sources.
    Retourne un DataFrame avec colonnes : midpointMjdTai, psfFlux,
    psfFluxErr, band, diaSourceId.  Retourne un DataFrame vide si aucune donnée.
    """
    payload = {
        "diaObjectId"  : str(dia_object_id),
        "columns"      : "r:midpointMjdTai,r:psfFlux,r:psfFluxErr,r:band,r:diaSourceId",
        "output-format": "json",
    }
    resp = requests.post(f"{BASE_URL}/api/v1/sources", json=payload, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    if not data:
        return pd.DataFrame()
    df = pd.DataFrame(data)
    df.columns = [c.replace('r:', '') for c in df.columns]
    for col in ['midpointMjdTai', 'psfFlux', 'psfFluxErr']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


def flux_to_mag_ab(flux_njy, flux_err_njy):
    """Flux (nJy) → magnitude AB + erreur propagée. NaN si flux ≤ 0."""
    flux   = np.asarray(flux_njy,     dtype=float)
    flux_e = np.asarray(flux_err_njy, dtype=float)
    valid   = flux > 0
    mag     = np.full(flux.shape, np.nan)
    mag_err = np.full(flux.shape, np.nan)
    mag[valid]     = -2.5 * np.log10(flux[valid]) + ZP_AB
    mag_err[valid] = (2.5 / np.log(10)) * np.abs(flux_e[valid] / flux[valid])
    return mag, mag_err


def _prepare_lc(df):
    """Nettoyage commun : conversion types, drop NaN, ajout colonne date."""
    df = df.copy()
    for col in ['midpointMjdTai', 'psfFlux', 'psfFluxErr']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna(subset=['midpointMjdTai', 'psfFlux', 'psfFluxErr'])
    df['date'] = mjd_to_datetime(df['midpointMjdTai'])
    return df


def _apply_date_axis(ax, show_xlabel=False):
    """Formattage commun de l'axe X (dates)."""
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=35, ha='right')
    ax.grid(True, alpha=0.2, linewidth=0.5, linestyle='--')
    if show_xlabel:
        ax.set_xlabel("Date (UTC)")


def _bands_in_order(lc):
    """Retourne les bandes présentes dans lc, dans l'ordre LSST ugrizy."""
    present = set(lc['band'].dropna().unique()) if 'band' in lc.columns else set()
    return [b for b in FILTER_ORDER if b in present]


def plot_lc_flux(ax, lc):
    """Panneau flux PSF (nJy) — toutes bandes superposées."""
    required = {'midpointMjdTai', 'psfFlux', 'psfFluxErr', 'band'}
    if not required.issubset(lc.columns):
        ax.text(0.5, 0.5, "Données manquantes", ha='center', va='center',
                transform=ax.transAxes, color='gray')
        return
    df = _prepare_lc(lc)
    if df.empty:
        ax.text(0.5, 0.5, "Pas de données valides", ha='center', va='center',
                transform=ax.transAxes, color='gray')
        return
    bands = _bands_in_order(df)
    for band in bands:
        sub   = df[df['band'] == band].sort_values('date')
        color = FILTER_COLORS.get(band, '#888888')
        ax.errorbar(
            sub['date'], sub['psfFlux'], yerr=sub['psfFluxErr'],
            fmt='o', color=color, label=f"${band}$",
            markersize=5, linewidth=0.9,
            capsize=3, capthick=0.8, elinewidth=0.8, alpha=0.88)
    ax.axhline(0, color='gray', lw=0.7, ls=':', alpha=0.6)
    ax.set_ylabel("Flux PSF (nJy)")
    ax.legend(
        ncol=len(bands), loc='upper left',
        framealpha=0.85, handlelength=1.2,
        handletextpad=0.4, columnspacing=0.8)
    _apply_date_axis(ax, show_xlabel=False)


def plot_lc_mag(ax, lc):
    """Panneau magnitude AB — toutes bandes superposées."""
    required = {'midpointMjdTai', 'psfFlux', 'psfFluxErr', 'band'}
    if not required.issubset(lc.columns):
        ax.text(0.5, 0.5, "Données manquantes", ha='center', va='center',
                transform=ax.transAxes, color='gray')
        return
    df = _prepare_lc(lc)
    if df.empty:
        ax.text(0.5, 0.5, "Pas de données valides", ha='center', va='center',
                transform=ax.transAxes, color='gray')
        return
    mag, mag_err = flux_to_mag_ab(df['psfFlux'].values, df['psfFluxErr'].values)
    df = df.copy()
    df['mag']     = mag
    df['mag_err'] = mag_err
    bands = _bands_in_order(df)
    for band in bands:
        sub   = df[df['band'] == band].sort_values('date')
        color = FILTER_COLORS.get(band, '#888888')
        det = sub.dropna(subset=['mag'])
        if not det.empty:
            ax.errorbar(
                det['date'], det['mag'], yerr=det['mag_err'],
                fmt='o', color=color, label=f"${band}$",
                markersize=5, linewidth=0.9,
                capsize=3, capthick=0.8, elinewidth=0.8, alpha=0.88)
        nondet = sub[sub['mag'].isna()]
        if not nondet.empty:
            sigma = nondet['psfFluxErr'].values
            v     = sigma > 0
            lim   = np.full(sigma.shape, np.nan)
            lim[v] = -2.5 * np.log10(3 * sigma[v]) + ZP_AB
            ax.scatter(
                nondet['date'].values, lim,
                marker='v', color=color, s=30,
                alpha=0.45, zorder=3, label='_nolegend_')
    ax.invert_yaxis()
    ax.set_ylabel("Magnitude AB")
    ax.legend(
        ncol=len(bands), loc='upper left',
        framealpha=0.85, handlelength=1.2,
        handletextpad=0.4, columnspacing=0.8)
    _apply_date_axis(ax, show_xlabel=True)


def make_lc_figure(oid, lc, extra_title=""):
    """
    Crée et retourne une figure courbe de lumière (flux haut / mag bas).
    Axe X partagé (sharex=True), figure (13x8).
    extra_title : texte supplémentaire ajouté au titre (ex. séparation).
    """
    lc_sorted = lc.dropna(subset=['midpointMjdTai']).sort_values(
        'midpointMjdTai', ascending=False)
    if not lc_sorted.empty:
        last       = lc_sorted.iloc[0]
        last_band  = last.get('band', 'N/A')
        last_date  = mjd_to_datestr(last.get('midpointMjdTai'))
        n_src      = len(lc)
        bands_used = ', '.join(_bands_in_order(lc)) or 'N/A'
    else:
        last_band = last_date = 'N/A'
        n_src = 0
        bands_used = 'N/A'

    main_title = (
        f"diaObjectId : {oid}{extra_title}\n"
        f"{n_src} sources   bandes : {bands_used}   "
        f"dernière alerte : {last_date}   filtre : {last_band}"
    )
    fig, (ax_flux, ax_mag) = plt.subplots(
        2, 1,
        figsize=(13, 8),
        sharex=True,
        layout='constrained',
        gridspec_kw={'height_ratios': [1, 1]},
    )
    fig.suptitle(main_title, fontsize=12, fontweight='bold')
    plot_lc_flux(ax_flux, lc)
    plot_lc_mag(ax_mag,  lc)
    plt.setp(ax_flux.xaxis.get_majorticklabels(), visible=False)
    return fig


# ── plot_cutouts conservé inchangé (spécifique conesearch) ──────────────────
def plot_cutouts(axes_row, src_id, kinds=CUTOUT_KINDS, cmap=CUTOUT_CMAP):
    arrays, pairs = [], []
    for ax, kind in zip(axes_row, kinds):
        try:
            params = {"diaSourceId": str(src_id), "kind": kind,
                      "output-format": CUTOUT_FORMAT}
            resp = requests.get(f"{BASE_URL}/api/v1/cutouts",
                                params=params, timeout=60)
            resp.raise_for_status()
            arr = np.array(Image.open(BytesIO(resp.content))).astype(float)
            if arr.ndim == 3: arr = arr.mean(axis=2)
            arrays.append(arr)
            im = ax.imshow(arr, cmap=cmap, origin='upper', aspect='auto')
            ax.axis('off')
            pairs.append((ax, im, arr))
        except Exception:
            ax.axis('off')
            ax.text(0.5, 0.5, f"{kind}\nerreur", ha='center', va='center',
                    transform=ax.transAxes, fontsize=8, color='red')
    if arrays:
        norm = plt.Normalize(vmin=min(a.min() for a in arrays),
                             vmax=max(a.max() for a in arrays))
        for _, im, _ in pairs: im.set_norm(norm)
    return pairs


print("Fonctions de tracé définies ✓")


In [ ]:
if df_cone.empty:
    print("⚠️  Aucun objet à afficher.")
else:
    for idx_obj, oid in enumerate(object_ids[:MAX_OBJECTS):

        # ── Récupération courbe de lumière via fetch_lightcurve ───────────
        lc = fetch_lightcurve(oid)
        if lc.empty:
            print(f"⚠️  {oid} — pas de données sources")
            continue

        # Séparation angulaire de cet objet
        sep_arcsec = df_cone[
            df_cone['diaObjectId'].astype(str) == str(oid)
        ]['separation_degree'].iloc[0] * 3600
        extra_title = f"   |   séparation : {sep_arcsec:.1f}\""

        # ── Courbe de lumière (layout ddf : flux haut / mag bas) ──────────
        if SHOW_LIGHTCURVES:
            fig_lc = make_lc_figure(oid, lc, extra_title=extra_title)

            if SAVE_FIGS:
                stem = f"{OUTPUT_DIR}/conesearch_obj_{oid}"
                fig_lc.savefig(f"{stem}.pdf", bbox_inches='tight', dpi=150)
                fig_lc.savefig(f"{stem}.png", bbox_inches='tight', dpi=150)

            plt.show()

        # ── Cutouts (figure séparée) ───────────────────────────────────────
        if SHOW_CUTOUTS:
            lc_sorted   = lc.dropna(subset=['midpointMjdTai']).sort_values(
                'midpointMjdTai', ascending=False)
            last_row    = lc_sorted.iloc[0] if not lc_sorted.empty else {}
            last_src_id = last_row.get('diaSourceId', None)
            last_band   = last_row.get('band', 'N/A')
            last_date   = mjd_to_datestr(last_row.get('midpointMjdTai'))

            if last_src_id is not None:
                fig_cut, axes_cut = plt.subplots(
                    1, 3, figsize=(13, 4), layout='constrained')
                fig_cut.suptitle(
                    f"diaObjectId : {oid}  |  cutouts  "
                    f"diaSourceId : {last_src_id}  "
                    f"filtre : {last_band}  date : {last_date}",
                    fontsize=10, fontweight='bold')
                pairs = plot_cutouts(axes_cut, last_src_id)
                for ax_c, kind in zip(axes_cut, CUTOUT_KINDS):
                    ax_c.set_title(f"{kind}\n{last_band}  {last_date}", fontsize=9)
                if pairs:
                    norm_c = plt.Normalize(
                        vmin=min(a.min() for _, _, a in pairs),
                        vmax=max(a.max() for _, _, a in pairs))
                    for _, im, _ in pairs: im.set_norm(norm_c)

                if SAVE_FIGS:
                    stem_cut = f"{OUTPUT_DIR}/conesearch_obj_{oid}_cutouts"
                    fig_cut.savefig(f"{stem_cut}.pdf", bbox_inches='tight', dpi=150)
                    fig_cut.savefig(f"{stem_cut}.png", bbox_inches='tight', dpi=150)

                plt.show()

        # ── Lien portail ──────────────────────────────────────────────────
        fink_url = f"https://lsst.fink-portal.org/{oid}"
        ipy_display(HTML(
            f'<div style="font-size:12px;margin:2px 0 14px 4px;">'
            f'🔗 <b>Portail Fink/LSST</b> — diaObjectId <code>{oid}</code> : '
            f'<a href="{fink_url}" target="_blank">{fink_url}</a></div>'
        ))
        print()

    print("\n✅ Affichage terminé.")


## 7. Tableau récapitulatif

In [ ]:
if not df_cone.empty:
    # Une ligne par objet (dernière alerte)
    id_col = next(c for c in df_cone.columns if 'diaObjectId' in c)
    df_last = (
        df_cone.sort_values('midpointMjdTai', ascending=False)
               .groupby(id_col, as_index=False).first()
    )
    df_last['last_date'] = df_last['midpointMjdTai'].apply(mjd_to_datestr)
    df_last['sep_arcsec'] = (df_last['separation_degree'] * 3600).round(2)
    df_last['psfFlux']    = df_last['psfFlux'].round(1)
    df_last['ra']         = df_last['ra'].round(6)
    df_last['dec']        = df_last['dec'].round(6)

    df_last['Portail Fink'] = df_last[id_col].apply(
        lambda oid: f'<a href="https://lsst.fink-portal.org/{oid}" target="_blank">'
                    f'🔗 {oid}</a>'
    )

    cols_show = [id_col, 'ra', 'dec', 'sep_arcsec',
                 'band', 'psfFlux', 'last_date', 'Portail Fink']
    cols_show = [c for c in cols_show if c in df_last.columns]
    df_show   = df_last[cols_show].sort_values('sep_arcsec')

    html = df_show.to_html(index=False, escape=False, border=0, classes='fink-table')
    style = """
    <style>
      .fink-table { border-collapse: collapse; font-size: 12px; width: 100%; }
      .fink-table th { background: #f0f0f0; padding: 6px 10px;
                       border-bottom: 2px solid #ccc; text-align: left; }
      .fink-table td { padding: 4px 10px; border-bottom: 1px solid #eee;
                       text-align: left; white-space: nowrap; }
      .fink-table tr:hover td { background: #fafafa; }
    </style>
    """
    ipy_display(HTML(style + html))